In [1]:
# Detailed train/val label quality audit
# Shows: total days, days with labels, days all-NaN, fire pixel stats
import os, glob, rasterio
import numpy as np

DATA_ROOT = "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire"
PATCH = 256

VAL_IDS = ["20568194","20701026","20562846","20700973","24462610",
           "24462788","24462753","24103571","21998313","21751303",
           "22141596","21999381","22712904"]

all_ids = sorted(os.listdir(DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]
train_ids = [d for d in numeric_ids if d not in VAL_IDS]
val_ids_found = [d for d in numeric_ids if d in VAL_IDS]

def audit_fire(fid):
    fdir = os.path.join(DATA_ROOT, fid)
    day_tifs = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))
    if not day_tifs:
        return {"id": fid, "days": 0, "has_b7": False, "days_ok": 0,
                "days_nan": 0, "fire_days": 0, "total_fire_px": 0, "status": "NO_FILES"}

    with rasterio.open(day_tifs[0]) as src:
        n_bands = src.count
        H, W = src.height, src.width

    if n_bands < 7:
        return {"id": fid, "days": len(day_tifs), "has_b7": False, "days_ok": 0,
                "days_nan": len(day_tifs), "fire_days": 0, "total_fire_px": 0,
                "status": "NO_BAND7"}

    r0 = (H - PATCH) // 2 if H >= PATCH else 0
    c0 = (W - PATCH) // 2 if W >= PATCH else 0
    ph = min(PATCH, H); pw = min(PATCH, W)

    days_ok = 0
    days_nan = 0
    fire_days = 0
    total_fire_px = 0

    for tif in day_tifs:
        with rasterio.open(tif) as src:
            b7 = src.read(7).astype(np.float32)
        if np.isnan(b7).sum() == b7.size:
            days_nan += 1
        else:
            days_ok += 1
            crop = b7[r0:r0+ph, c0:c0+pw]
            fpx = int((crop >= 7).sum())
            total_fire_px += fpx
            if fpx > 0:
                fire_days += 1

    n = len(day_tifs)
    if days_ok == 0:
        status = "ZERO_LABELS"
    elif days_nan == 0:
        status = "FULL"
    elif days_ok >= n * 0.5:
        status = "PARTIAL_OK"
    else:
        status = "MOSTLY_NAN"

    return {"id": fid, "days": n, "has_b7": True, "days_ok": days_ok,
            "days_nan": days_nan, "fire_days": fire_days,
            "total_fire_px": total_fire_px, "status": status}


# ---- Audit train ----
print(f"Auditing {len(train_ids)} TRAIN fires...")
train_results = []
for i, fid in enumerate(train_ids):
    r = audit_fire(fid)
    train_results.append(r)
    if (i+1) % 30 == 0 or (i+1) == len(train_ids):
        print(f"\r  {i+1}/{len(train_ids)}", end="", flush=True)
print()

# ---- Audit val ----
print(f"Auditing {len(val_ids_found)} VAL fires...")
val_results = []
for fid in val_ids_found:
    r = audit_fire(fid)
    val_results.append(r)

# ---- Summary tables ----
def print_summary(results, label):
    full = [r for r in results if r["status"] == "FULL"]
    partial_ok = [r for r in results if r["status"] == "PARTIAL_OK"]
    mostly_nan = [r for r in results if r["status"] == "MOSTLY_NAN"]
    zero = [r for r in results if r["status"] == "ZERO_LABELS"]
    no_files = [r for r in results if r["status"] in ("NO_FILES", "NO_BAND7")]

    total = len(results)
    print(f"\n{'='*70}")
    print(f"{label} SUMMARY ({total} fires)")
    print(f"{'='*70}")
    print(f"  FULL labels (all days OK):     {len(full):3d} ({len(full)/total*100:.0f}%)")
    print(f"  PARTIAL (>50% days OK):         {len(partial_ok):3d} ({len(partial_ok)/total*100:.0f}%)")
    print(f"  MOSTLY NaN (<50% days OK):      {len(mostly_nan):3d} ({len(mostly_nan)/total*100:.0f}%)")
    print(f"  ZERO labels (all days NaN):     {len(zero):3d} ({len(zero)/total*100:.0f}%)")
    print(f"  No files/no band7:              {len(no_files):3d} ({len(no_files)/total*100:.0f}%)")

    usable = full + partial_ok
    print(f"\n  USABLE for training:            {len(usable):3d} ({len(usable)/total*100:.0f}%)")
    print(f"  EXCLUDE from training:          {len(mostly_nan)+len(zero)+len(no_files):3d}")

    # Show excluded fires
    excluded = mostly_nan + zero + no_files
    if excluded:
        print(f"\n  Fires to EXCLUDE:")
        for r in sorted(excluded, key=lambda x: x["id"]):
            print(f"    {r['id']}: {r['status']} "
                  f"({r['days_ok']}/{r['days']} days OK, "
                  f"{r['total_fire_px']} fire px)")

    # Show partial fires with details
    if partial_ok:
        print(f"\n  PARTIAL fires (usable but with some NaN days):")
        for r in sorted(partial_ok, key=lambda x: x["days_nan"], reverse=True)[:10]:
            pct = r['days_ok'] / max(r['days'], 1) * 100
            print(f"    {r['id']}: {r['days_ok']}/{r['days']} days OK ({pct:.0f}%), "
                  f"{r['fire_days']} fire days, {r['total_fire_px']} fire px")

    return [r["id"] for r in excluded]


train_exclude = print_summary(train_results, "TRAIN")
val_exclude = print_summary(val_results, "VAL")

# ---- Combined exclude list ----
all_exclude = sorted(set(train_exclude + val_exclude))
print(f"\n{'='*70}")
print(f"TOTAL FIRES TO EXCLUDE: {len(all_exclude)}")
print(f"{'='*70}")
print(f"NO_LABEL_IDS = [")
for eid in all_exclude:
    print(f'    "{eid}",')
print(f"]")
print(f"\nCopy this list into Config.NO_LABEL_IDS for clean training.")

Auditing 138 TRAIN fires...
  138/138
Auditing 13 VAL fires...

TRAIN SUMMARY (138 fires)
  FULL labels (all days OK):      54 (39%)
  PARTIAL (>50% days OK):          63 (46%)
  MOSTLY NaN (<50% days OK):        3 (2%)
  ZERO labels (all days NaN):       4 (3%)
  No files/no band7:               14 (10%)

  USABLE for training:            117 (85%)
  EXCLUDE from training:           21

  Fires to EXCLUDE:
    20777207: NO_FILES (0/0 days OK, 0 fire px)
    20777386: NO_FILES (0/0 days OK, 0 fire px)
    21693566: ZERO_LABELS (0/20 days OK, 0 fire px)
    21751305: MOSTLY_NAN (4/9 days OK, 1422 fire px)
    21751309: ZERO_LABELS (0/11 days OK, 0 fire px)
    21889672: NO_FILES (0/0 days OK, 0 fire px)
    21889683: NO_FILES (0/0 days OK, 0 fire px)
    21889697: NO_FILES (0/0 days OK, 0 fire px)
    21889719: NO_FILES (0/0 days OK, 0 fire px)
    21889734: NO_FILES (0/0 days OK, 0 fire px)
    21889754: NO_FILES (0/0 days OK, 0 fire px)
    21890056: MOSTLY_NAN (2/9 days OK, 1065 fire